## Import libraries

In [1]:
import csv
import pandas as pd
import config

In [2]:
# Define Paths
YAHOO_RAW_DIR = config.RAW_DATA_DIR / "yahoo"
FRED_RAW_DIR = config.RAW_DATA_DIR / "fred"
FAMA_FRENCH_RAW_DIR = config.RAW_DATA_DIR / "fama_french"

YAHOO_INPUT_DIR = config.INPUT_DATA_DIR / "yahoo"
FRED_INPUT_DIR = config.INPUT_DATA_DIR / "fred"
FAMA_FRENCH_INPUT_DIR = config.INPUT_DATA_DIR / "fama_french"

YAHOO_INPUT_DIR.mkdir(parents=True, exist_ok=True)
FRED_INPUT_DIR.mkdir(parents=True, exist_ok=True)
FAMA_FRENCH_INPUT_DIR.mkdir(parents=True, exist_ok=True)

# Read the master investable universe
universe = pd.read_csv(config.UNIVERSE_FILE)

## Functions

In [3]:
def clean_yahoo_file(input_file, output_file):

    # Read the raw file
    data = pd.read_csv(input_file)

    # Convert Date
    data["Date"] = pd.to_datetime(data["Date"])

    # Get the Yahoo ticker from the filename
    ticker = input_file.stem

    # Check and remove rows with missing values
    missing_rows = data.isna().any(axis=1).sum()

    if missing_rows > 0:
        data = data.dropna()
        print(f"{ticker}: {missing_rows} rows with missing values were found and removed.")

    # Check and remove duplicate dates
    duplicate_rows = data["Date"].duplicated(keep="last").sum()

    if duplicate_rows > 0:
        data = data.drop_duplicates(subset="Date", keep="last")
        print(f"{ticker}: {duplicate_rows} duplicate dates were found and removed.")

    # Sort chronologically
    data = data.sort_values("Date")

    # Save cleaned file
    data.to_csv(output_file, index=False)

    return data


def clean_fred_file(input_file, output_file):

    # Read the raw file
    data = pd.read_csv(input_file)

    # Convert Date
    data["Date"] = pd.to_datetime(data["Date"])

    # Get the FRED series name from the filename
    series_name = input_file.stem

     # Check and remove rows with missing values
    missing_rows = data.isna().any(axis=1).sum()

    if missing_rows > 0:
        data = data.dropna()
        print(f"{series_name}: {missing_rows} rows with missing values were found and removed.")

    # Check and remove duplicate dates
    duplicate_rows = data["Date"].duplicated(keep="last").sum()

    if duplicate_rows > 0:
        data = data.drop_duplicates(subset="Date", keep="last")
        print(f"{series_name}: {duplicate_rows} duplicate dates were found and removed.")

    # Sort chronologically
    data = data.sort_values("Date")

    # Save cleaned file
    data.to_csv(output_file, index=False)

    return data


def clean_fama_french_file(input_file, output_file):
    """
    Clean a Kenneth French CSV file by:
    - removing metadata rows before the actual header
    - using the existing column names
    - adding 'Date' as the first column name
    - removing non-data rows after the observations
    - converting Date to datetime
    """

    # Read the raw file line by line
    with open(input_file, "r", encoding="latin1") as f:
        rows = list(csv.reader(f))

    # Find the first row where the first cell is a YYYYMMDD date
    data_row = None

    for i, row in enumerate(rows):
        if len(row) > 0:
            first_cell = row[0].strip()

            if first_cell.isdigit() and len(first_cell) == 8 and first_cell.startswith("19"):
                data_row = i
                break

    if data_row is None:
        raise ValueError(f"Could not identify the first data row in {input_file}")

    # The row immediately above contains the header
    header_row = data_row - 1

    # Get the existing column names from the header
    columns = rows[header_row]

    # Rename only the first column
    columns[0] = "Date"

    # Keep rows from the first data row onward
    data_rows = rows[data_row:]

    # Keep only rows whose first cell is a valid YYYYMMDD date
    data_rows = [
        row for row in data_rows
        if len(row) > 0
        and row[0].strip().isdigit()
        and len(row[0].strip()) == 8
    ]

    # Create DataFrame
    data = pd.DataFrame(data_rows, columns=columns)

    # Convert Date to datetime
    data["Date"] = pd.to_datetime(data["Date"], format="%Y%m%d")

    # Get the file name
    file_name = input_file.stem

    # Check and remove rows with missing values
    missing_rows = data.isna().any(axis=1).sum()

    if missing_rows > 0:
        data = data.dropna()
        print(f"{file_name}: {missing_rows} rows with missing values were found and removed.")

    # Check and remove duplicate dates
    duplicate_rows = data["Date"].duplicated(keep="last").sum()

    if duplicate_rows > 0:
        data = data.drop_duplicates(subset="Date", keep="last")
        print(f"{file_name}: {duplicate_rows} duplicate dates were found and removed.")

    # Sort chronologically
    data = data.sort_values("Date")

    # Save cleaned file
    data.to_csv(output_file, index=False)

    return data


def audit_yahoo_data(input_dir, output_file):

    audit_records = []

    for input_file in input_dir.glob("*.csv"):

        data = pd.read_csv(input_file)
        data["Date"] = pd.to_datetime(data["Date"])
        ticker = input_file.stem

        audit_records.append({
            "Ticker": ticker,
            "Start Date": data["Date"].min().date(),
            "End Date": data["Date"].max().date(),
            "Observations": len(data),
            "Missing OHLC": data[["Open", "High", "Low", "Close"]].isna().any(axis=1).sum(),
            "Missing Adj Close": data["Adj Close"].isna().sum(),
            "Missing Volume": data["Volume"].isna().sum(),
            "Duplicate Dates": data["Date"].duplicated().sum()
        })

    audit_data = pd.DataFrame(audit_records)
    audit_data.to_csv(output_file, index=False)

    return audit_data


def audit_fred_data(input_dir, output_file):

    audit_records = []

    for input_file in input_dir.glob("*.csv"):

        data = pd.read_csv(input_file)
        data["Date"] = pd.to_datetime(data["Date"])
        series_name = input_file.stem

        audit_records.append({
            "Series": series_name,
            "Start Date": data["Date"].min().date(),
            "End Date": data["Date"].max().date(),
            "Observations": len(data),
            "Missing Values": data.isna().any(axis=1).sum(),
            "Duplicate Dates": data["Date"].duplicated().sum()
        })

    audit_data = pd.DataFrame(audit_records)
    audit_data.to_csv(output_file, index=False)
    
    return audit_data


def audit_fama_french_data(input_dir, output_file):

    audit_records = []

    for input_file in input_dir.glob("*.csv"):

        data = pd.read_csv(input_file)
        data["Date"] = pd.to_datetime(data["Date"])

        file_name = input_file.stem

        audit_records.append({
            "File": file_name,
            "Start Date": data["Date"].min().date(),
            "End Date": data["Date"].max().date(),
            "Observations": len(data),
            "Missing Values": data.isna().any(axis=1).sum(),
            "Duplicate Dates": data["Date"].duplicated().sum()
        })

    audit_data = pd.DataFrame(audit_records)

    audit_data.to_csv(output_file, index=False)

    return audit_data

## Yahoo Finance Files

In [4]:
for input_file in YAHOO_RAW_DIR.glob("*.csv"):
    clean_yahoo_file(
        input_file=input_file,
        output_file=YAHOO_INPUT_DIR / input_file.name
    )

print("\nYahoo Finance data cleaning is completed.")


Yahoo Finance data cleaning is completed.


## FRED Files

In [5]:
for input_file in FRED_RAW_DIR.glob("*.csv"):
    clean_fred_file(
        input_file=input_file,
        output_file=FRED_INPUT_DIR / input_file.name
    )

print("\nFRED data cleaning is completed.")

AAA10Y: 479 rows with missing values were found and removed.
BAA10Y: 443 rows with missing values were found and removed.
CPIAUCSL: 1 rows with missing values were found and removed.
DCOILWTICO: 374 rows with missing values were found and removed.
DGS10: 720 rows with missing values were found and removed.
DGS2: 552 rows with missing values were found and removed.
DGS30: 545 rows with missing values were found and removed.
DGS3MO: 492 rows with missing values were found and removed.
DTWEXBGS: 211 rows with missing values were found and removed.
T10Y2Y: 552 rows with missing values were found and removed.
T10Y3M: 484 rows with missing values were found and removed.
T10YIE: 254 rows with missing values were found and removed.
T5YIE: 254 rows with missing values were found and removed.
T5YIFR: 254 rows with missing values were found and removed.
THREEFYTP10: 414 rows with missing values were found and removed.
THREEFYTP2: 414 rows with missing values were found and removed.
THREEFYTP5: 41

## Fama-French Files

In [6]:
# 5-factor file
ff5_data = clean_fama_french_file(
    input_file=FAMA_FRENCH_RAW_DIR / "F-F_Research_Data_5_Factors_2x3_daily.csv",
    output_file=FAMA_FRENCH_INPUT_DIR / "Fama_French_5_Factors_Daily.csv"
)

# Momentum file
mom_data = clean_fama_french_file(
    input_file=FAMA_FRENCH_RAW_DIR / "F-F_Momentum_Factor_daily.csv",
    output_file=FAMA_FRENCH_INPUT_DIR / "Momentum_Factor_Daily.csv"
)

print("\nFama-French data cleaning is completed.")


Fama-French data cleaning is completed.


## Data Audit

In [7]:
# Yahoo
yahoo_audit = audit_yahoo_data(
    input_dir=YAHOO_INPUT_DIR,
    output_file=config.INPUT_DATA_DIR / "yahoo_data_audit.csv"
)

# FRED
fred_audit = audit_fred_data(
    input_dir=FRED_INPUT_DIR,
    output_file=config.INPUT_DATA_DIR / "fred_data_audit.csv"
)

# Fama-French
fama_french_audit = audit_fama_french_data(
    input_dir=FAMA_FRENCH_INPUT_DIR,
    output_file=config.INPUT_DATA_DIR / "fama_french_data_audit.csv"
)